### Import modules and data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

# Setting up the path to include the parent directory
sys.path.append(str(Path.cwd().parent.parent))
from backend.config.settings import PATHS
from backend.data.cleaning import clean_string_series
from sklearn.metrics.pairwise import haversine_distances

In [ ]:
detailed_path = PATHS['raw_data_notebooks'] / 'UTF8list-3.tsv'
detailed_reservoirs_pd = pd.read_csv(detailed_path, sep='\t')
detailed_reservoirs_pd.head()

In [ ]:
detailed_reservoirs_pd.info()

### Renaming columns

In [ ]:
# Will assign them to their lowercase version
detailed_reservoirs_pd.columns = detailed_reservoirs_pd.columns.str.lower()
columns_dict = {'x': 'longitude', 'y': 'latitude'}
detailed_reservoirs_pd.rename(columns=columns_dict, inplace=True)

### Looking for Missing Values before cleaning strings and converting dtypes

In [ ]:
detailed_reservoirs_pd.isna().sum()

### Cleaning string data

- Without missing values:

In [ ]:
def clean_string_columns(detailed_reservoirs_data, columns):
    for column in columns:
        detailed_reservoirs_data.loc[:, column] = clean_string_series(detailed_reservoirs_data[column])

clean_string_columns(detailed_reservoirs_pd, ['name', 'reservoir', 'basin', 'province', 'autonomous_community', 'type'])

- With missing values:

In [ ]:
detailed_reservoirs_pd.loc[detailed_reservoirs_pd['riverbed'].notna(), 'riverbed'] = clean_string_series(detailed_reservoirs_pd.loc[detailed_reservoirs_pd['riverbed'].notna(), 'riverbed'])

In [ ]:
detailed_reservoirs_pd.head()

### Converting coordinate columns

In [ ]:
detailed_reservoirs_pd['longitude'] = detailed_reservoirs_pd['longitude'].str.replace(',', '.').astype(float)
detailed_reservoirs_pd['latitude'] = detailed_reservoirs_pd['latitude'].str.replace(',', '.').astype(float)

### Converting other columns to float

In [ ]:
detailed_reservoirs_pd.loc[detailed_reservoirs_pd['crest_elevation'].notna(), 'crest_elevation'] = detailed_reservoirs_pd.loc[detailed_reservoirs_pd['crest_elevation'].notna(), 'crest_elevation'].str.replace(',', '.').astype(float)
detailed_reservoirs_pd.loc[detailed_reservoirs_pd['dam_height'].notna(), 'dam_height'] = detailed_reservoirs_pd.loc[detailed_reservoirs_pd['dam_height'].notna(), 'dam_height'].str.replace(',', '.').astype(float)

In [ ]:
detailed_reservoirs_pd.head()

### Save current cleaning and developing detailed_reservoir.ipynb at EDA folder

In [ ]:
cleaned_detailed_reservoir_path = PATHS['pre_EDA'] / 'detailed_reservoir_for_EDA.csv'
cleaned_detailed_reservoir_path.parent.mkdir(parents=True, exist_ok=True)
detailed_reservoirs_pd.to_csv(cleaned_detailed_reservoir_path, index=False)

## Post EDA Analysis

### Deleting Code and Reservoir Columns

In [ ]:
detailed_reservoirs_pd = detailed_reservoirs_pd.drop(columns=['code', 'reservoir'])

### Handling Missing Values

In [ ]:
detailed_reservoirs_pd.head()

In [ ]:
detailed_reservoirs_pd.isna().sum()

#### Riverbed Missing Values

Sin nombre means without name, those are NaNs, as we said at detailed_reservoirs.ipynb from the EDA folder

In [ ]:
mask = detailed_reservoirs_pd['riverbed'] == 'sin nombre'
detailed_reservoirs_pd.loc[mask, 'riverbed'] = np.nan

In [ ]:
def impute_nearest_neighbour(detailed_reservoirs_data, column_name):
    nan_reservoirs = detailed_reservoirs_data[detailed_reservoirs_data[column_name].isna()]
    non_nan_reservoirs = detailed_reservoirs_data[detailed_reservoirs_data[column_name].notna()]

    nan_coord_rads = np.radians(nan_reservoirs[['longitude', 'latitude']].to_numpy()).reshape(-1, 2)
    non_nan_coord_rads = np.radians(non_nan_reservoirs[['longitude', 'latitude']].to_numpy()).reshape(-1, 2)

    distances = haversine_distances(nan_coord_rads, non_nan_coord_rads) * 6371.0 
    closest_indices = distances.argmin(axis=1)

    # Create a mapping from indices to reservoir values
    reservoir_mapping = non_nan_reservoirs[column_name].iloc[closest_indices]
    detailed_reservoirs_data.loc[nan_reservoirs.index, column_name] = reservoir_mapping.values
    return detailed_reservoirs_data

In [ ]:
detailed_reservoirs_pd = impute_nearest_neighbour(detailed_reservoirs_pd, 'riverbed')

### Crest Elevation Missing Values

In [ ]:
detailed_reservoirs_pd.isna().sum()

In [ ]:
detailed_reservoirs_pd.info()

In [ ]:
detailed_reservoirs_pd = impute_nearest_neighbour(detailed_reservoirs_pd, 'crest_elevation')

In [ ]:
# Now that we have the crest elevation filled, we can convert it to float
detailed_reservoirs_pd['crest_elevation'] = detailed_reservoirs_pd['crest_elevation'].astype(float)

### Removing duplicated rows

In [ ]:
# First sight:
mask_duplicated = detailed_reservoirs_pd['name'].value_counts() > 1
duplicated = detailed_reservoirs_pd.loc[detailed_reservoirs_pd['name'].map(mask_duplicated)].sort_values('name')
duplicated.head(10)

In [ ]:
# We well keep the first one that has the highest dam height
detailed_reservoirs_pd = detailed_reservoirs_pd.sort_values('dam_height', ascending=False).drop_duplicates('name')

### Dam height Column

- Missing values have been studied at EDA/detailed_reservoir.ipynb and it was decided to leave the missing values as they were.
- Several types of Random Forest Regressors were tried to impute missing values in the dam height column, but the predictions turned out to be random.

In [ ]:
# We convert non-nan dam height to float
mask = detailed_reservoirs_pd['dam_height'].isna() == False
detailed_reservoirs_pd.loc[mask, 'dam_height'] = detailed_reservoirs_pd.loc[mask, 'dam_height'].replace(',', '.').astype(float)

In [ ]:
detailed_reservoirs_pd.info()

In [ ]:
cleaned_detailed_reservoirs_path = PATHS['cleaned_data_notebooks'] / 'detailed_reservoirs_cleaned.csv'
cleaned_detailed_reservoirs_path.parent.mkdir(parents=True, exist_ok=True)
detailed_reservoirs_pd.to_csv(cleaned_detailed_reservoirs_path, index=False)